In [ ]:
!pip install "transformers==4.36.0" --q

In [ ]:
!pip install datasets==2.19.0 --q

In [ ]:
import pandas as pd
import torch
from datasets import load_dataset
import transformers
from transformers import AutoTokenizer, AutoConfig, AutoModel

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

### Loading a dataset

In [ ]:
dataset = load_dataset(
    "emarro/clinvar_vep_512",
    split="test"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print(dataset)

Dataset({
    features: ['ref_forward_sequence', 'alt_forward_sequence', 'label', 'chromosome', 'position', 'seq', 'ref', 'alt', 'start', 'stop', 'chrom'],
    num_rows: 1018
})


In [ ]:
df = pd.DataFrame(dataset)
df.head()

,ref_forward_sequence,alt_forward_sequence,label,chromosome,position,seq,ref,alt,start,stop,chrom
0,ACTGGGGTAGTGATGTGTCCTTGTTTCTATCGAGTCAACACAAAAT...,ACTGGGGTAGTGATGTGTCCTTGTTTCTATCGAGTCAACACAAAAT...,1,8,1771124,ACTGGGGTAGTGATGTGTCCTTGTTTCTATCGAGTCAACACAAAAT...,C,G,1771124,1771636,8
1,CCTTGTTTCTATCGAGTCAACACAAAATGAATGATCTTTCTTTTAG...,CCTTGTTTCTATCGAGTCAACACAAAATGAATGATCTTTCTTTTAG...,1,8,1771142,CCTTGTTTCTATCGAGTCAACACAAAATGAATGATCTTTCTTTTAG...,G,C,1771142,1771654,8
2,TCTCCATGCAGGCGGGCTGGTCCGAGTCTCTGTTTTGGAAGCTCAA...,TCTCCATGCAGGCGGGCTGGTCCGAGTCTCTGTTTTGGAAGCTCAA...,1,8,1780495,TCTCCATGCAGGCGGGCTGGTCCGAGTCTCTGTTTTGGAAGCTCAA...,G,C,1780495,1781007,8
3,GGGAACATGTTTCACCATGTCCAGGAAGGGCTGTATTTTTAAGGTC...,GGGAACATGTTTCACCATGTCCAGGAAGGGCTGTATTTTTAAGGTC...,1,8,1882669,GGGAACATGTTTCACCATGTCCAGGAAGGGCTGTATTTTTAAGGTC...,C,T,1882669,1883181,8
4,TAACACTTCAGAGGGTTTTTAGAGGGAGTAGATGTGAACTTGTGTG...,TAACACTTCAGAGGGTTTTTAGAGGGAGTAGATGTGAACTTGTGTG...,1,8,6414797,TAACACTTCAGAGGGTTTTTAGAGGGAGTAGATGTGAACTTGTGTG...,C,G,6414797,6415309,8


### Loading and manipulating a dataset
https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised

In [ ]:
dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks_revised")
print(dataset)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 493242
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 38822
    })
})


This gives us a Huggingface DatasetDict, containing two Huggingface Dataset objects, one for the train and one for the test split.

In [ ]:
train_dataset = dataset["train"]
print(train_dataset)

Dataset({
    features: ['sequence', 'name', 'label', 'task'],
    num_rows: 493242
})


A Huggingface Dataset can be converted into a Pandas DataFrame.

In [ ]:
train_df = train_dataset.to_pandas()
train_df.head()

,sequence,name,label,task
0,ATATTTTTCGGTGTTTTTTTAAAATCCAGAAAAGGTTAATTGTTTT...,chr8:16774000-16775000|0,0,H2AFZ
1,AAGTATAGTCATAAAGGTGGGGGATTTAATCATGTTCCAGAATGAT...,chr12:12784548-12785548|1,1,H2AFZ
2,CCCCAGCATGGACTTGCAGGGGCGCCTGGTGGGAGAGGGGTGTGGA...,chr1:1376174-1377174|1,1,H2AFZ
3,GCCTGTAATCCCAGCACTTTGAGAGGCCGGGGCGGGCAGATCACGA...,chr17:78711000-78712000|0,0,H2AFZ
4,CTGCGGCTTTTGGAGGGAGCTTGGCTTGGAACCCAGGGTTCTCTGA...,chr8:127981888-127982888|1,1,H2AFZ


In [ ]:
train_df = pd.DataFrame(train_dataset)
train_df.head()

,sequence,name,label,task
0,ATATTTTTCGGTGTTTTTTTAAAATCCAGAAAAGGTTAATTGTTTT...,chr8:16774000-16775000|0,0,H2AFZ
1,AAGTATAGTCATAAAGGTGGGGGATTTAATCATGTTCCAGAATGAT...,chr12:12784548-12785548|1,1,H2AFZ
2,CCCCAGCATGGACTTGCAGGGGCGCCTGGTGGGAGAGGGGTGTGGA...,chr1:1376174-1377174|1,1,H2AFZ
3,GCCTGTAATCCCAGCACTTTGAGAGGCCGGGGCGGGCAGATCACGA...,chr17:78711000-78712000|0,0,H2AFZ
4,CTGCGGCTTTTGGAGGGAGCTTGGCTTGGAACCCAGGGTTCTCTGA...,chr8:127981888-127982888|1,1,H2AFZ


You can then do all the usual Pandas manipulations, which can be helpful for inspecting the dataset.

In [ ]:
train_df[train_df["task"]=="H2AFZ"]

,sequence,name,label,task
0,ATATTTTTCGGTGTTTTTTTAAAATCCAGAAAAGGTTAATTGTTTT...,chr8:16774000-16775000|0,0,H2AFZ
1,AAGTATAGTCATAAAGGTGGGGGATTTAATCATGTTCCAGAATGAT...,chr12:12784548-12785548|1,1,H2AFZ
2,CCCCAGCATGGACTTGCAGGGGCGCCTGGTGGGAGAGGGGTGTGGA...,chr1:1376174-1377174|1,1,H2AFZ
3,GCCTGTAATCCCAGCACTTTGAGAGGCCGGGGCGGGCAGATCACGA...,chr17:78711000-78712000|0,0,H2AFZ
4,CTGCGGCTTTTGGAGGGAGCTTGGCTTGGAACCCAGGGTTCTCTGA...,chr8:127981888-127982888|1,1,H2AFZ
...,...,...,...,...
29995,TATTAATATCTGACTAAGGAATGGGCTCTTTCTTCTCCCACTCCTA...,chr3:139204486-139205486|1,1,H2AFZ
29996,ATTCAGGTTTCATTTGTTAAATCTGCAAGCTCTGTATGCTGTCTTT...,chr2:145237926-145238926|1,1,H2AFZ
29997,GATATGAACTTTGTTGACGAATGTGTTCTCTGCACAGCTAACCCAG...,chr14:101051000-101052000|0,0,H2AFZ
29998,GTTATTAGGCACAAGACACTTCTTTTAGGCACAAGACACTTCTTTT...,chr16:71240000-71241000|0,0,H2AFZ


In [ ]:
train_df[train_df["task"]=="H2AFZ"]["label"].value_counts()

,count
label,
0,15003
1,14997


In [ ]:
train_df["task"].value_counts()

,count
task,
H2AFZ,30000
H3K27ac,30000
H3K27me3,30000
H3K36me3,30000
H3K4me1,30000
H3K4me2,30000
H4K20me1,30000
promoter_all,30000
enhancers_types,30000


The Huggingface Dataset has its own `train_test_split` function which can be used to further divide the dataset.

In [ ]:
split_dataset = train_dataset.train_test_split(test_size=0.1)
split_dataset

DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 443917
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 49325
    })
})

This produces another DatasetDict, containing two Datasets - one "train" and one "test".

Other Dataset functions include `select`:

In [ ]:
small_dataset = train_dataset.select(range(50))
print(small_dataset)
print(len(small_dataset))

Dataset({
    features: ['sequence', 'name', 'label', 'task'],
    num_rows: 50
})
50


And `filter`:

In [ ]:
filtered_dataset = train_dataset.filter(lambda x: x["task"]=="H2AFZ")
print(filtered_dataset)

Filter:   0%|          | 0/493242 [00:00<?, ? examples/s]

Dataset({
    features: ['sequence', 'name', 'label', 'task'],
    num_rows: 30000
})


In [ ]:
filtered_df = filtered_dataset.to_pandas()
filtered_df["task"].value_counts()

,count
task,
H2AFZ,30000


### What is a tokeniser?

* The tokeniser is used to prepare the data for model input.
* It breaks a sequence into smaller chunks or "tokens".

Related Huggingface pages:
* https://huggingface.co/docs/transformers/en/main_classes/tokenizer
* https://huggingface.co/learn/llm-course/chapter2/4

### How does the Nucleotide Transformer v2 tokeniser work?
* This tokeniser splits sequences into 6-mers - chunks of 6 nucleotides - when possible, otherwise tokenising each nucleotide separately.
* This tokenizer has a vocabulary size of 4105. The inputs of the model are then of the form:
```
<CLS> <ACGTGT> <ACGTGC> <ACGGAC> <GACTAG> <TCAGCA>
```
* The tokenised sequences have a maximum length of 1,000.

Related Huggingface page: https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species


In [ ]:
MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"

In [ ]:
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

In [ ]:
# https://training.galaxyproject.org/training-material/topics/statistics/tutorials/genomic-llm-finetuning/tutorial.html
tokeniser = transformers.AutoTokenizer.from_pretrained(
    MODEL_NAME,
    model_max_length=512,
    padding_side="right",
    use_fast=True,
    trust_remote_code=True,
)

### Example: Tokenise an example sequence for input into Nucleotide Transformer v2

Let's get an example sequence from the dataset

In [ ]:
sequence = train_dataset["sequence"][0]
print(sequence)
print(len(sequence))

ATATTTTTCGGTGTTTTTTTAAAATCCAGAAAAGGTTAATTGTTTTTTATATTGACCATGTTTTCTGTAAATTCTTTCATGCTTTATCTTTTTTCTTTGCATTATCCGAGTTATTTCCAGCCTGTTTGTCATTGGTTTAATTTTCAACAATGCCTACCTCACTATTTTTTATTAGTTCCAATATTATGTCTTTTTATACTCAATTCATTTCCCTAGTTCTCCCATACTATTCTCTGGTCTGTTATCTTTTCATCTTTGAGTTTTGGTTTTATATAGTCTATACACTTTATAATTTCTTTAAAAATGCAAAGCATTAGTTGTGAATGTTTTTCTTTTGTTTTCTAGGGCGTGTTCTCATTTAGAGTACTACGCATGAGAAGGTCATTTATTTCATTTTTTAAATAATCATAGGAACTTCTCATAGTTGTTATCCTGTCTCTTCTCATTTCACTCATATTAGACAAGGCAGCTCTTCTCTTTGCCCATGTGTGAGCTCTCCTTGAATCTCTTCTTATCTGCGATCTAAGAGTGTTGTCTTACCTTCAGATGGTCAACGTGAAGAATGGGTATTGATAGGTCACACTCACTCTGCTCTTTGTTTATCTGAGATTGAGAAGCTAGAGAGAGGAAGCTATGGGCAGTTAGGAAGAAGTTGGTCTCTTCTCTATTTTCCTATTGATTTTATAAAATCCTGTTCTGTATTTTGTTTTTTCTAGTGAAATGTTGACTTCTCTCCTTAATTCAGAGTGAGCTGCTGAATCTTCCCTCTACAAAATATCACTCATTCTTACTAGGCTTTATTCCTCTTTCCACTGAATATTGAGCTTCTAAGTTCAAATGGGAGTCTTTCAGGCTAGCCATTAACTTTTCCTGTTCTGTAAATGCCAAGATAAAATATTATGAGAGATTAAATTAGTAAATCTAGTTAGATTTTTTCTTATAATGAAAATACAAGCTGGAAAATTGGAGAATGCTGTACAACCCTTCCAAACTCCAACAT

By passing the sequence into the tokeniser, we get the input ids and attention mask.

In [ ]:
tok = tokeniser(sequence)
print(tok)

{'input_ids': [3, 283, 1475, 3419, 1286, 1682, 67, 1053, 1371, 285, 653, 1375, 3335, 1628, 491, 1131, 1375, 1511, 1137, 855, 1449, 2683, 1895, 2011, 91, 2086, 1962, 2664, 1115, 1306, 3438, 282, 1899, 1354, 2439, 1563, 2713, 1440, 2127, 364, 2015, 3404, 1374, 1629, 859, 3931, 281, 1610, 2203, 267, 1626, 13, 2068, 339, 1910, 1883, 1627, 3420, 1284, 3548, 1563, 826, 2356, 466, 990, 1355, 1563, 1350, 1054, 1270, 2412, 315, 3404, 2527, 2412, 350, 2442, 1336, 254, 3691, 2459, 3751, 3554, 3692, 2422, 1643, 2380, 1975, 2322, 3549, 1624, 2409, 508, 189, 199, 4055, 1817, 3470, 2447, 2540, 1403, 1133, 797, 788, 1234, 3318, 3661, 3987, 1270, 3131, 3948, 1439, 348, 2333, 346, 1031, 2683, 2519, 1403, 1375, 886, 477, 607, 2475, 94, 3298, 3711, 3103, 1708, 1158, 286, 2443, 2383, 1003, 1120, 1628, 2210, 285, 924, 1083, 2061, 3900, 1385, 3668, 2134, 2396, 2524, 1862, 1958, 3142, 282, 1849, 326, 1338, 106, 3410, 1371, 2378, 454, 294, 3714, 29, 3271, 3706, 2096, 1446, 622, 4102, 4104, 4102, 4103], 'attent

In [ ]:
token_ids = tok["input_ids"]
print(token_ids)

[3, 283, 1475, 3419, 1286, 1682, 67, 1053, 1371, 285, 653, 1375, 3335, 1628, 491, 1131, 1375, 1511, 1137, 855, 1449, 2683, 1895, 2011, 91, 2086, 1962, 2664, 1115, 1306, 3438, 282, 1899, 1354, 2439, 1563, 2713, 1440, 2127, 364, 2015, 3404, 1374, 1629, 859, 3931, 281, 1610, 2203, 267, 1626, 13, 2068, 339, 1910, 1883, 1627, 3420, 1284, 3548, 1563, 826, 2356, 466, 990, 1355, 1563, 1350, 1054, 1270, 2412, 315, 3404, 2527, 2412, 350, 2442, 1336, 254, 3691, 2459, 3751, 3554, 3692, 2422, 1643, 2380, 1975, 2322, 3549, 1624, 2409, 508, 189, 199, 4055, 1817, 3470, 2447, 2540, 1403, 1133, 797, 788, 1234, 3318, 3661, 3987, 1270, 3131, 3948, 1439, 348, 2333, 346, 1031, 2683, 2519, 1403, 1375, 886, 477, 607, 2475, 94, 3298, 3711, 3103, 1708, 1158, 286, 2443, 2383, 1003, 1120, 1628, 2210, 285, 924, 1083, 2061, 3900, 1385, 3668, 2134, 2396, 2524, 1862, 1958, 3142, 282, 1849, 326, 1338, 106, 3410, 1371, 2378, 454, 294, 3714, 29, 3271, 3706, 2096, 1446, 622, 4102, 4104, 4102, 4103]


In [ ]:
attention_mask = tok["attention_mask"]
print(attention_mask)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


By using the `tokenize` function, we get the tokens themselves.

In [ ]:
tokens = tokeniser.tokenize(sequence)
print(tokens)

['ATATTT', 'TTCGGT', 'GTTTTT', 'TTAAAA', 'TCCAGA', 'AAAGGT', 'TAATTG', 'TTTTTT', 'ATATTG', 'ACCATG', 'TTTTCT', 'GTAAAT', 'TCTTTC', 'ATGCTT', 'TATCTT', 'TTTTCT', 'TTGCAT', 'TATCCG', 'AGTTAT', 'TTCCAG', 'CCTGTT', 'TGTCAT', 'TGGTTT', 'AATTTT', 'CAACAA', 'TGCCTA', 'CCTCAC', 'TATTTT', 'TTATTA', 'GTTCCA', 'ATATTA', 'TGTCTT', 'TTTATA', 'CTCAAT', 'TCATTT', 'CCCTAG', 'TTCTCC', 'CATACT', 'ATTCTC', 'TGGTCT', 'GTTATC', 'TTTTCA', 'TCTTTG', 'AGTTTT', 'GGTTTT', 'ATATAG', 'TCTATA', 'CACTTT', 'ATAATT', 'TCTTTA', 'AAAATG', 'CAAAGC', 'ATTAGT', 'TGTGAA', 'TGTTTT', 'TCTTTT', 'GTTTTC', 'TAGGGC', 'GTGTTC', 'TCATTT', 'AGAGTA', 'CTACGC', 'ATGAGA', 'AGGTCA', 'TTTATT', 'TCATTT', 'TTTAAA', 'TAATCA', 'TAGGAA', 'CTTCTC', 'ATAGTT', 'GTTATC', 'CTGTCT', 'CTTCTC', 'ATTTCA', 'CTCATA', 'TTAGAC', 'AAGGCA', 'GCTCTT', 'CTCTTT', 'GCCCAT', 'GTGTGA', 'GCTCTC', 'CTTGAA', 'TCTCTT', 'CTTATC', 'TGCGAT', 'CTAAGA', 'GTGTTG', 'TCTTAC', 'CTTCAG', 'ATGGTC', 'AACGTG', 'AAGAAT', 'GGGTAT', 'TGATAG', 'GTCACA', 'CTCACT', 'CTGCTC', 'TTTGTT',

To decode the sequence, we can put the input ids into the `decode` function.

In [ ]:
decoded = tokeniser.decode(token_ids)
print(decoded)

<cls> ATATTT TTCGGT GTTTTT TTAAAA TCCAGA AAAGGT TAATTG TTTTTT ATATTG ACCATG TTTTCT GTAAAT TCTTTC ATGCTT TATCTT TTTTCT TTGCAT TATCCG AGTTAT TTCCAG CCTGTT TGTCAT TGGTTT AATTTT CAACAA TGCCTA CCTCAC TATTTT TTATTA GTTCCA ATATTA TGTCTT TTTATA CTCAAT TCATTT CCCTAG TTCTCC CATACT ATTCTC TGGTCT GTTATC TTTTCA TCTTTG AGTTTT GGTTTT ATATAG TCTATA CACTTT ATAATT TCTTTA AAAATG CAAAGC ATTAGT TGTGAA TGTTTT TCTTTT GTTTTC TAGGGC GTGTTC TCATTT AGAGTA CTACGC ATGAGA AGGTCA TTTATT TCATTT TTTAAA TAATCA TAGGAA CTTCTC ATAGTT GTTATC CTGTCT CTTCTC ATTTCA CTCATA TTAGAC AAGGCA GCTCTT CTCTTT GCCCAT GTGTGA GCTCTC CTTGAA TCTCTT CTTATC TGCGAT CTAAGA GTGTTG TCTTAC CTTCAG ATGGTC AACGTG AAGAAT GGGTAT TGATAG GTCACA CTCACT CTGCTC TTTGTT TATCTG AGATTG AGAAGC TAGAGA GAGGAA GCTATG GGCAGT TAGGAA GAAGTT GGTCTC TTCTCT ATTTTC CTATTG ATTTTA TAAAAT CCTGTT CTGTAT TTTGTT TTTTCT AGTGAA ATGTTG ACTTCT CTCCTT AATTCA GAGTGA GCTGCT GAATCT TCCCTC TACAAA ATATCA CTCATT CTTACT AGGCTT TATTCC TCTTTC CACTGA ATATTG AGCTTC TAAGTT CAAATG GGAGTC TTTCAG 

### Try it yourself: Tokenise an example sequence for DNABERT-2
Huggingface page: https://huggingface.co/zhihan1996/DNABERT-2-117M

In [ ]:
# Get the model name
MODEL_NAME =

In [ ]:
# Load the tokeniser
# Use a maximum length of 512bp
# Pad on the left side only


In [ ]:
# Get the tokenised output

In [ ]:
# Get the input_ids

In [ ]:
# Decode the input_ids

#### Answer

In [ ]:
MODEL_NAME = "zhihan1996/DNABERT-2-117M"

In [ ]:
# Load the tokeniser
# Use a maximum length of 512bp
# Pad on the left side only
tokeniser = transformers.AutoTokenizer.from_pretrained(
    MODEL_NAME,
    model_max_length=512,
    padding_side="left",
    use_fast=True,
    trust_remote_code=True,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Get the tokenised output
tokens = tokeniser.tokenize(sequence)
print(tokens)

['A', 'TATTTT', 'TCGGTG', 'TTTTTT', 'TAAAA', 'TCCA', 'GAAAA', 'GG', 'TTAATT', 'GTTTT', 'TTATATT', 'GA', 'CCATG', 'TTTTCTG', 'TAAATT', 'CTT', 'TCATG', 'CTT', 'TATC', 'TTTTTT', 'CTTTGCA', 'TTATCC', 'GA', 'GTTATT', 'TCCAGCC', 'TGTTTG', 'TCATT', 'GGTT', 'TAATTTT', 'CAA', 'CAATG', 'CCTA', 'CCTCA', 'CTATTTT', 'TTATTA', 'GTTCCAA', 'TATTATG', 'TCTTTT', 'TATA', 'CTCAATT', 'CATT', 'TCCCTA', 'GTTCTCC', 'CATA', 'CTATT', 'CTCTG', 'GTCTG', 'TTA', 'TCTTTT', 'CATCTT', 'TGAGTTTT', 'GGTTTTA', 'TATA', 'GTCTATA', 'CACTT', 'TATAATT', 'TCTT', 'TAAAAATG', 'CAAA', 'GCATTA', 'GTT', 'GTGAA', 'TGTTTT', 'TCTTTT', 'GTTTT', 'CTA', 'GGGC', 'GTGTT', 'CTCA', 'TTTA', 'GAGTA', 'CTA', 'CGCA', 'TGA', 'GAAGG', 'TCATT', 'TATTTCA', 'TTTTTTAAA', 'TAA', 'TCATA', 'GGAA', 'CTTCTCA', 'TAGTT', 'GTTA', 'TCCTG', 'TCTCTT', 'CTCATT', 'TCACTCA', 'TATTA', 'GACAA', 'GGCA', 'GCTCTT', 'CTCTTTG', 'CCCA', 'TGTGTGA', 'GC', 'TCTCCTT', 'GAA', 'TCTCTT', 'CTTA', 'TCTG', 'CGA', 'TCTAA', 'GAGTGTT', 'GTCTTA', 'CCTT', 'CAGATG', 'GTCAA', 'CGTGAA', 'GA

In [ ]:
# Get the input_ids
input_ids = tokeniser(sequence)["input_ids"]
print(input_ids)

[1, 5, 155, 1848, 122, 58, 82, 85, 15, 453, 87, 1337, 17, 249, 1354, 244, 29, 224, 29, 276, 122, 2400, 3648, 17, 381, 3331, 341, 127, 101, 399, 27, 238, 114, 190, 854, 273, 2532, 1892, 167, 46, 1498, 51, 942, 2468, 104, 186, 178, 220, 24, 167, 734, 3085, 2070, 46, 2666, 219, 1871, 47, 1379, 49, 754, 31, 135, 1261, 167, 87, 37, 354, 163, 63, 94, 282, 37, 159, 23, 259, 127, 887, 1345, 20, 434, 57, 1533, 237, 77, 177, 271, 432, 1890, 147, 225, 112, 471, 1020, 95, 3498, 19, 816, 25, 271, 81, 54, 93, 172, 2965, 697, 64, 521, 168, 1063, 2899, 43, 1462, 411, 63, 178, 149, 687, 126, 73, 145, 105, 50, 534, 772, 112, 77, 1147, 316, 271, 40, 155, 929, 1302, 58, 1552, 28, 155, 509, 37, 327, 48, 226, 596, 453, 2726, 62, 119, 47, 372, 1572, 211, 432, 81, 37, 269, 43, 559, 82, 119, 43, 831, 79, 31, 373, 33, 161, 289, 37, 19, 313, 91, 705, 28, 318, 76, 1243, 147, 1505, 364, 24, 120, 88, 77, 2165, 81, 3542, 196, 62, 618, 33, 223, 28, 196, 232, 131, 415, 12, 8, 2]


In [ ]:
# Decode the input_ids
decoded = tokeniser.decode(input_ids)
print(decoded)

[CLS] A TATTTT TCGGTG TTTTTT TAAAA TCCA GAAAA GG TTAATT GTTTT TTATATT GA CCATG TTTTCTG TAAATT CTT TCATG CTT TATC TTTTTT CTTTGCA TTATCC GA GTTATT TCCAGCC TGTTTG TCATT GGTT TAATTTT CAA CAATG CCTA CCTCA CTATTTT TTATTA GTTCCAA TATTATG TCTTTT TATA CTCAATT CATT TCCCTA GTTCTCC CATA CTATT CTCTG GTCTG TTA TCTTTT CATCTT TGAGTTTT GGTTTTA TATA GTCTATA CACTT TATAATT TCTT TAAAAATG CAAA GCATTA GTT GTGAA TGTTTT TCTTTT GTTTT CTA GGGC GTGTT CTCA TTTA GAGTA CTA CGCA TGA GAAGG TCATT TATTTCA TTTTTTAAA TAA TCATA GGAA CTTCTCA TAGTT GTTA TCCTG TCTCTT CTCATT TCACTCA TATTA GACAA GGCA GCTCTT CTCTTTG CCCA TGTGTGA GC TCTCCTT GAA TCTCTT CTTA TCTG CGA TCTAA GAGTGTT GTCTTA CCTT CAGATG GTCAA CGTGAA GAATGGG TATT GATAGG TCACA CTCA CTCTG CTCTT TGTTTA TCTGA GATT GAGAA GCTA GAGA GAGGAA GCTATG GGCA GTTA GGAAGAA GTTGG TCTCTT CTC TATTTT CCTATT GATTTTA TAAAA TCCTGTT CTG TATTTT GTTTTTT CTA GTGAAA TGTT GACTT CTCTCC TTAATT CAGAGTGA GCTG CTGAA TCTT CCCTC TACAAAA TATCA CTCATT CTTA CTA GGCTT TATT CCTCTT TCCA CTGAA TATT GAGCTT CTAA G

### Example: Tokenise a list of sequences for Nucleotide Transformer v2
Huggingface page: https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species

In [ ]:
MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"

tokeniser = transformers.AutoTokenizer.from_pretrained(
    MODEL_NAME,
    model_max_length=512,
    padding_side="right",
    use_fast=True,
    trust_remote_code=True,
)

Let's select the first five sequences in the dataset as our batch.

In [ ]:
batch = train_dataset["sequence"][0:5]
print(batch)

['ATATTTTTCGGTGTTTTTTTAAAATCCAGAAAAGGTTAATTGTTTTTTATATTGACCATGTTTTCTGTAAATTCTTTCATGCTTTATCTTTTTTCTTTGCATTATCCGAGTTATTTCCAGCCTGTTTGTCATTGGTTTAATTTTCAACAATGCCTACCTCACTATTTTTTATTAGTTCCAATATTATGTCTTTTTATACTCAATTCATTTCCCTAGTTCTCCCATACTATTCTCTGGTCTGTTATCTTTTCATCTTTGAGTTTTGGTTTTATATAGTCTATACACTTTATAATTTCTTTAAAAATGCAAAGCATTAGTTGTGAATGTTTTTCTTTTGTTTTCTAGGGCGTGTTCTCATTTAGAGTACTACGCATGAGAAGGTCATTTATTTCATTTTTTAAATAATCATAGGAACTTCTCATAGTTGTTATCCTGTCTCTTCTCATTTCACTCATATTAGACAAGGCAGCTCTTCTCTTTGCCCATGTGTGAGCTCTCCTTGAATCTCTTCTTATCTGCGATCTAAGAGTGTTGTCTTACCTTCAGATGGTCAACGTGAAGAATGGGTATTGATAGGTCACACTCACTCTGCTCTTTGTTTATCTGAGATTGAGAAGCTAGAGAGAGGAAGCTATGGGCAGTTAGGAAGAAGTTGGTCTCTTCTCTATTTTCCTATTGATTTTATAAAATCCTGTTCTGTATTTTGTTTTTTCTAGTGAAATGTTGACTTCTCTCCTTAATTCAGAGTGAGCTGCTGAATCTTCCCTCTACAAAATATCACTCATTCTTACTAGGCTTTATTCCTCTTTCCACTGAATATTGAGCTTCTAAGTTCAAATGGGAGTCTTTCAGGCTAGCCATTAACTTTTCCTGTTCTGTAAATGCCAAGATAAAATATTATGAGAGATTAAATTAGTAAATCTAGTTAGATTTTTTCTTATAATGAAAATACAAGCTGGAAAATTGGAGAATGCTGTACAACCCTTCCAAACTCCAAC

The tokeniser and its `decode` function can natively handle batches.

In [ ]:
encoded_batch = tokeniser(batch)
print(encoded_batch)

In [ ]:
decoded_batch = tokeniser.decode(encoded_batch["input_ids"])
print(decoded_batch)

The `tokenize` function does not have a native batch mode; instead, it process one string at a time.

In [ ]:
tokens = tokeniser.tokenize(batch)
print(tokens)

TypeError: unhashable type: 'list'

Instead:

In [ ]:
tokens = [tokeniser.tokenize(seq) for seq in batch]
print(tokens)

[['ATATTT', 'TTCGGT', 'GTTTTT', 'TTAAAA', 'TCCAGA', 'AAAGGT', 'TAATTG', 'TTTTTT', 'ATATTG', 'ACCATG', 'TTTTCT', 'GTAAAT', 'TCTTTC', 'ATGCTT', 'TATCTT', 'TTTTCT', 'TTGCAT', 'TATCCG', 'AGTTAT', 'TTCCAG', 'CCTGTT', 'TGTCAT', 'TGGTTT', 'AATTTT', 'CAACAA', 'TGCCTA', 'CCTCAC', 'TATTTT', 'TTATTA', 'GTTCCA', 'ATATTA', 'TGTCTT', 'TTTATA', 'CTCAAT', 'TCATTT', 'CCCTAG', 'TTCTCC', 'CATACT', 'ATTCTC', 'TGGTCT', 'GTTATC', 'TTTTCA', 'TCTTTG', 'AGTTTT', 'GGTTTT', 'ATATAG', 'TCTATA', 'CACTTT', 'ATAATT', 'TCTTTA', 'AAAATG', 'CAAAGC', 'ATTAGT', 'TGTGAA', 'TGTTTT', 'TCTTTT', 'GTTTTC', 'TAGGGC', 'GTGTTC', 'TCATTT', 'AGAGTA', 'CTACGC', 'ATGAGA', 'AGGTCA', 'TTTATT', 'TCATTT', 'TTTAAA', 'TAATCA', 'TAGGAA', 'CTTCTC', 'ATAGTT', 'GTTATC', 'CTGTCT', 'CTTCTC', 'ATTTCA', 'CTCATA', 'TTAGAC', 'AAGGCA', 'GCTCTT', 'CTCTTT', 'GCCCAT', 'GTGTGA', 'GCTCTC', 'CTTGAA', 'TCTCTT', 'CTTATC', 'TGCGAT', 'CTAAGA', 'GTGTTG', 'TCTTAC', 'CTTCAG', 'ATGGTC', 'AACGTG', 'AAGAAT', 'GGGTAT', 'TGATAG', 'GTCACA', 'CTCACT', 'CTGCTC', 'TTTGTT'

### Example: Tokenise a dataset for Nucleotide Transformer v2
First, let's try to tokenise all the sequences in the `sequence` column.

In [ ]:
seq = train_dataset["sequence"]
print(seq)

Buffered data was truncated after reaching the output size limit.

In [ ]:
seq_tok = tokeniser(list(seq)) # The column needs to be converted to a list in order to be passed directly to the tokeniser
print(seq_tok)

Buffered data was truncated after reaching the output size limit.

However, this is not the most efficient way to do things! When working with large datasets or tokenising multiple columns, it is preferable to use the `.map` function.

This is a built-in function of the Huggingface dataset object, and efficiently handles batches and multiprocessing. This approach avoids loading all sequences into memory at once.

In [39]:
type(train_dataset)

datasets.arrow_dataset.Dataset

We first define a tokenise function.

In [40]:
def tokenise_batch(batch):
    return tokeniser(
        batch["sequence"], # Select only the sequence column
        padding=True,
        truncation=True,
        max_length=512,
    )

We then apply this to the dataset using `.map`.

In [ ]:
tokenised_dataset = dataset.map(tokenise_batch, batched=True, batch_size=64)
print(tokenised_dataset)

### Try it yourself: Tokenise all the sequences in the test split of the dataset

In [ ]:
# Get only the test section of the dataset
test_dataset = # Your code here
print(test_dataset)

In [ ]:
# Define the tokenise function

In [ ]:
# Tokenise all the sequences in the column

In [ ]:
# Decode the sequences

#### Answer

In [ ]:
# Get only the test section of the dataset
test_dataset = dataset["test"]
print(test_dataset)

In [ ]:
# Define the tokenise function
def tokenise_batch(batch):
    return tokeniser(
        batch["sequence"], # Select only the ref_forward_sequence column
        padding=True,
        truncation=True,
        max_length=512,
    )

In [ ]:
# Tokenise all the sequences in the column
tokenised_dataset = dataset.map(tokenise_batch, batched=True, batch_size=64)
print(tokenised_dataset)

In [ ]:
# Decode the sequences

# Method 1: list
decoded = tokeniser.decode(list(tokenised_dataset["input_ids"]))
print(decoded)

In [ ]:
# Method 2: map
def decode_batch(batch):
  return {"tokens": tokeniser.decode(batch["input_ids"])}

decoded = tokenised_dataset.map(decode_batch, batched=True, batch_size=64)
print(decoded)
print(decoded["tokens"])